# Single-photoelectron calibration with LED pulses

The AD3 drives the LED with a free-running train of 100 ns pulses; the tee'd copy on Rigol Ch3 is the
trigger. The Rigol records triggered frames in hardware and reads them back afterwards.

1. **Parameters** (edit below), then **Connect**: checks the PMT HV is on and stable (it never changes HV;
   use `python -m ad3pulse hv set ...` for that).
2. **Timing**: mean waveforms at a brighter setting, to check the signal window covers the light.
3. **Occupancy scan** per PMT: mean photoelectrons per pulse, μ, against LED amplitude.
4. **Choose amplitude** per PMT for μ ≈ 0.15 (1 photon per ~7 pulses).
5. **Calibration run** per PMT at its amplitude, saved raw to `runs/spe/`.
6. **SPE fit**: pedestal + Poisson-weighted 1/2/3-PE Gaussians → SPE charge and gain.

**Occupancy** uses zero-PE counting: μ = −ln P0(signal window) + ln P0(control window), where P0 is the
fraction of pulses whose charge is below a threshold (pedestal + N_SIGMA σ). The control window has the
same width but sits before the trigger, so it measures dark counts and noise only.

The two PMTs need different light levels (Ch1 is far more sensitive), so each gets its own scan range and run.

In [ ]:
import numpy as np

# ---- Instruments -----------------------------------------------------------------
SCOPE_RESOURCE = "192.168.0.101"
HV_RESOURCE = "192.168.0.100"
HV_EXPECTED_V = {0: 1550.0, 1: 875.0}   # HV channel -> expected |V|; checked, never changed
HV_TOLERANCE_V = 2.0

PMTS = {                                 # Rigol channel -> label
    1: "PMT A (HV ch0, -1550 V)",
    2: "PMT B (HV ch1, -875 V)",
}
PULSER_CHANNEL = 3                       # tee'd copy of the LED drive; trigger source

# ---- LED pulses (AD3 W1) -----------------------------------------------------------
PULSE_WIDTH_S = 100e-9
PULSE_RATE_HZ = 1000.0
MAX_LED_V = 3.0                          # hard limit on the AD3 amplitude
SCAN_AMPLITUDES_V = {                    # AD3 setting per PMT (Ch3 reads ~5% lower)
    1: np.round(np.arange(1.50, 1.801, 0.02), 3),
    2: np.round(np.arange(1.70, 2.001, 0.02), 3),
}
SCAN_STOP_MU = 1.0                       # stop a PMT's scan once μ exceeds this
TIMING_AMPLITUDE_V = {1: 1.80, 2: 2.20}  # bright enough to see the light arrival

# ---- Scope -------------------------------------------------------------------------
PMT_SCALE_V_DIV = 0.010                  # Ch1/Ch2 at 50 Ohm; ±40 mV on screen
PULSER_SCALE_V_DIV = 0.5
PULSER_OFFSET_V = -1.5                   # shows -0.5..3.5 V on Ch3
TIMEBASE_S_DIV = 100e-9
TIME_OFFSET_S = 0.0                      # record spans -500..+500 ns around the trigger
MEMORY_DEPTH = "1k"                      # 1000 points at 1 GS/s
TRIG_LEVEL_V = 0.5                       # rising edge on Ch3

# ---- Analysis windows (seconds from the trigger) -------------------------------------
BASELINE_WINDOW = (-500e-9, -300e-9)
CONTROL_WINDOW = (-280e-9, -80e-9)       # same width as the signal window, no LED light
SIGNAL_WINDOW = (40e-9, 240e-9)          # light arrives ~80-180 ns
BASELINE_PER_FRAME = False               # False: one baseline per run (adds no noise)
N_SIGMA = 3.0                            # zero-PE threshold above the pedestal
TARGET_MU = 0.15
TARGET_RANGE = (0.10, 0.20)

# ---- Run sizes ---------------------------------------------------------------------
SCAN_FRAMES = 500                        # per scan point (~80 s each)
CAL_FRAMES = 20000                       # per PMT calibration run
CHUNK_FRAMES = 5000                      # frames per recording / saved file
OUTPUT_DIR = "runs/spe"
SCOPE_RETRIES = 3                        # reconnect attempts if the scope drops the LAN link

: 

In [ ]:
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pyvisa

from ad3pulse.hardware import FrameSet, IsegHV, RigolScope
from ad3pulse.instrument import AnalogDiscovery, PulseSpec
from ad3pulse.spe import baseline_subtract, fit_spe, integrate_charge, occupancy, pick_amplitude

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# HV: check only.
with IsegHV(HV_RESOURCE) as hv:
    hv_state = {}
    for ch, expected in HV_EXPECTED_V.items():
        hv.channel = ch
        status, v, i = hv.read_status(), hv.read_voltage(), hv.read_current()
        hv_state[ch] = {"V": v, "I_A": i}
        print(f"HV ch{ch}: {v:.2f} V, {i * 1e6:.2f} uA, {', '.join(hv.describe_status(status))}")
        if not status & IsegHV.IS_ON or abs(abs(v) - expected) > HV_TOLERANCE_V:
            raise RuntimeError(f"HV ch{ch} is not on at {expected:g} V; set it with "
                               f"'python -m ad3pulse hv set {expected:g} --channel {ch}'")

def open_scope():
    """Connect to the Rigol and apply all acquisition settings."""
    s = RigolScope(SCOPE_RESOURCE).connect()
    for ch in PMTS:
        s.configure_channel(ch, PMT_SCALE_V_DIV, 0.0, impedance="FIFTy")
    s.configure_channel(PULSER_CHANNEL, PULSER_SCALE_V_DIV, PULSER_OFFSET_V, impedance="OMEG")
    s.configure_timebase(TIMEBASE_S_DIV, TIME_OFFSET_S, MEMORY_DEPTH)
    s.configure_edge_trigger(PULSER_CHANNEL, TRIG_LEVEL_V, "POS")
    s.set_trigger_sweep("NORM")
    s._write(":RUN")
    return s


scope = open_scope()
print(f"Scope: {scope.idn}; recorder holds {scope.max_record_frames()} frames")

ad3 = AnalogDiscovery().open()
PULSE = PulseSpec(amplitude=0.5, width=PULSE_WIDTH_S, pre_delay=100e-9, post_delay=1e-6)
CHANNELS = (*PMTS, PULSER_CHANNEL)


def record(amplitude, n_frames, label, channels=CHANNELS):
    """Set the LED amplitude, record n_frames triggers, save them raw, return the FrameSet."""
    ad3.set_pulse_amplitude(amplitude, max_v=MAX_LED_V)
    time.sleep(0.2)
    global scope
    for attempt in range(1, SCOPE_RETRIES + 1):
        try:
            fs = scope.record_frames(n_frames, channels, timeout_s=n_frames / PULSE_RATE_HZ + 30)
            break
        except (OSError, pyvisa.errors.VisaIOError) as exc:
            # The scope occasionally drops the LAN connection mid-readout; reconnect and redo it.
            print(f"    scope connection lost ({exc!r}); reconnecting, attempt {attempt}")
            try:
                scope.close()
            except Exception:
                pass
            time.sleep(10)
            scope = open_scope()
    else:
        raise RuntimeError(f"scope readout failed {SCOPE_RETRIES} times")
    t = fs.time_s
    fs.meta.update(
        label=label, amplitude_V=amplitude, width_s=PULSE_WIDTH_S, rate_Hz=PULSE_RATE_HZ,
        pmt_scale_V_div=PMT_SCALE_V_DIV, hv=hv_state, recorded_at=datetime.now().isoformat())
    if PULSER_CHANNEL in channels:
        window = (t > 20e-9) & (t < 80e-9)
        fs.meta["pulser_height_V"] = float(np.median(fs.volts(PULSER_CHANNEL)[:, window]))
    fs.save(out_dir / f"{label}_{amplitude:.3f}V_{stamp}.npz")
    return fs


def charges(fs, ch):
    """(signal, control) charge per frame in pC for scope channel ch."""
    v = baseline_subtract(fs.volts(ch), fs.time_s, BASELINE_WINDOW, BASELINE_PER_FRAME)
    return (integrate_charge(v, fs.time_s, SIGNAL_WINDOW),
            integrate_charge(v, fs.time_s, CONTROL_WINDOW))

: 

## Timing check
Mean waveforms at a brighter setting per PMT, with the windows shaded. The light pulse should sit inside
the green signal window and nothing should appear in the grey control window.

In [ ]:
fig, axes = plt.subplots(len(PMTS), 1, sharex=True, figsize=(9, 3 * len(PMTS)))
ad3.start_pulse_train(PULSE, PULSE_RATE_HZ)
try:
    for ax, (ch, label) in zip(np.atleast_1d(axes), PMTS.items()):
        fs = record(TIMING_AMPLITUDE_V[ch], 200, f"timing_ch{ch}")
        v = baseline_subtract(fs.volts(ch), fs.time_s, BASELINE_WINDOW, BASELINE_PER_FRAME)
        ax.plot(fs.time_s * 1e9, v.mean(0) * 1e3, lw=1, label=f"{label}, LED {TIMING_AMPLITUDE_V[ch]:g} V")
        for (lo, hi), color in ((SIGNAL_WINDOW, "C2"), (CONTROL_WINDOW, "0.6"), (BASELINE_WINDOW, "C0")):
            ax.axvspan(lo * 1e9, hi * 1e9, color=color, alpha=0.15)
        ax.set_ylabel("mean (mV)")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
finally:
    ad3.stop_output(0)
np.atleast_1d(axes)[-1].set_xlabel("time from trigger (ns)")
fig.tight_layout()
fig.savefig(out_dir / f"timing_{stamp}.png", dpi=150)
plt.show()

## Occupancy scan
For each PMT, step the LED amplitude and measure μ from SCAN_FRAMES pulses. A PMT's scan stops once μ
exceeds SCAN_STOP_MU. Every scan point is saved raw.

In [ ]:
scan = {ch: [] for ch in PMTS}
ad3.start_pulse_train(PULSE, PULSE_RATE_HZ)
try:
    for ch, label in PMTS.items():
        print(label)
        for amp in SCAN_AMPLITUDES_V[ch]:
            fs = record(float(amp), SCAN_FRAMES, f"scan_ch{ch}")
            occ = occupancy(*charges(fs, ch), n_sigma=N_SIGMA)
            scan[ch].append((float(amp), fs.meta["pulser_height_V"], occ))
            print(f"  LED {amp:.3f} V (Ch3 {fs.meta['pulser_height_V']:.3f} V): "
                  f"mu = {occ.mu:.4f}  (P0 {occ.p0_signal:.3f} vs control {occ.p0_control:.3f}, "
                  f"threshold {occ.threshold_pC:.3f} pC)")
            if occ.mu > SCAN_STOP_MU:
                break
finally:
    ad3.stop_output(0)

fig, ax = plt.subplots(figsize=(7, 4.5))
for ch, rows in scan.items():
    amps = [r[0] for r in rows]
    mus = [max(r[2].mu, 1e-4) for r in rows]
    errs = [np.sqrt((1 - r[2].p0_signal) / (r[2].n * max(r[2].p0_signal, 1e-9))) for r in rows]
    ax.errorbar(amps, mus, yerr=errs, marker="o", ms=4, capsize=2, label=PMTS[ch])
ax.axhspan(*TARGET_RANGE, color="C2", alpha=0.15, label="target")
ax.set_yscale("log")
ax.set_xlabel("AD3 LED amplitude (V)")
ax.set_ylabel("μ (PE per pulse)")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(out_dir / f"occupancy_scan_{stamp}.png", dpi=150)
plt.show()

## Choose the calibration amplitude
Interpolated per PMT for TARGET_MU. Override `CAL_AMPLITUDE_V` by hand if needed.

In [ ]:
CAL_AMPLITUDE_V = {}
for ch, rows in scan.items():
    amp = pick_amplitude([r[0] for r in rows], [r[2].mu for r in rows], TARGET_MU)
    if amp is None:
        print(f"{PMTS[ch]}: target mu {TARGET_MU} not inside the scanned range; widen SCAN_AMPLITUDES_V")
        continue
    CAL_AMPLITUDE_V[ch] = round(amp, 4)
    print(f"{PMTS[ch]}: LED {CAL_AMPLITUDE_V[ch]:.4f} V for mu = {TARGET_MU}")

## Calibration run
CAL_FRAMES pulses per PMT at its own amplitude, recorded in chunks of CHUNK_FRAMES and saved raw. Only
that PMT's channel is read (about 70 ms per frame). The achieved μ is printed after every chunk.

In [ ]:
cal = {}
ad3.start_pulse_train(PULSE, PULSE_RATE_HZ)
try:
    for ch, amp in CAL_AMPLITUDE_V.items():
        q_sig, q_ctrl = [], []
        for part in range(int(np.ceil(CAL_FRAMES / CHUNK_FRAMES))):
            n = min(CHUNK_FRAMES, CAL_FRAMES - part * CHUNK_FRAMES)
            t0 = time.monotonic()
            # Only this PMT's channel: halves the readout time of a long run.
            fs = record(amp, n, f"cal_ch{ch}_part{part:02d}", channels=(ch,))
            s, c = charges(fs, ch)
            q_sig.append(s)
            q_ctrl.append(c)
            occ = occupancy(np.concatenate(q_sig), np.concatenate(q_ctrl), n_sigma=N_SIGMA)
            print(f"{PMTS[ch]} part {part}: {n} frames in {time.monotonic() - t0:.0f} s; "
                  f"mu so far {occ.mu:.4f}")
        cal[ch] = (np.concatenate(q_sig), np.concatenate(q_ctrl), amp)
finally:
    ad3.stop_output(0)

## Optional: load saved runs instead (no hardware needed)
Skip this after a live calibration run. Run it (after the **Parameters** cell only) to rebuild the
calibration charges from files in `OUTPUT_DIR`, e.g. to re-fit with different analysis windows or on
another PC. Uses the newest `cal_ch<N>_part*.npz` run per PMT unless `LOAD_STAMP` is set.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from ad3pulse.hardware import FrameSet
from ad3pulse.spe import baseline_subtract, fit_spe, integrate_charge, occupancy

LOAD_STAMP = None                        # e.g. "20261007_114256"; None = newest run per PMT
out_dir = Path(OUTPUT_DIR)


def charges(fs, ch):
    """(signal, control) charge per frame in pC for scope channel ch."""
    v = baseline_subtract(fs.volts(ch), fs.time_s, BASELINE_WINDOW, BASELINE_PER_FRAME)
    return (integrate_charge(v, fs.time_s, SIGNAL_WINDOW),
            integrate_charge(v, fs.time_s, CONTROL_WINDOW))


cal = {}
for ch in PMTS:
    files = sorted(out_dir.glob(f"cal_ch{ch}_part*.npz"))
    if not files:
        print(f"{PMTS[ch]}: no cal_ch{ch}_part*.npz files in {out_dir}")
        continue
    run = LOAD_STAMP or max("_".join(f.stem.split("_")[-2:]) for f in files)
    files = [f for f in files if f.stem.endswith(run)]
    q_sig, q_ctrl = [], []
    for f in files:
        fs = FrameSet.load(f)
        s, c = charges(fs, ch)
        q_sig.append(s)
        q_ctrl.append(c)
    cal[ch] = (np.concatenate(q_sig), np.concatenate(q_ctrl), fs.meta["amplitude_V"])
    print(f"{PMTS[ch]}: run {run}, {len(files)} file(s), {len(cal[ch][0])} pulses, "
          f"LED {fs.meta['amplitude_V']} V")
stamp = f"refit_{run}"


## SPE fit
Signal-window charge spectrum (log scale) with the fitted pedestal + 1/2/3-PE model; the control-window
spectrum is shown for comparison.

In [ ]:
fits = {}
fig, axes = plt.subplots(len(cal), 1, figsize=(8, 3.5 * len(cal)), squeeze=False)
for ax, (ch, (q_sig, q_ctrl, amp)) in zip(axes[:, 0], cal.items()):
    occ = occupancy(q_sig, q_ctrl, n_sigma=N_SIGMA)
    fit = fit_spe(q_sig, mu_guess=occ.mu)
    fits[ch] = fit
    print(f"{PMTS[ch]} (LED {amp:.4f} V, {len(q_sig)} pulses): {fit}; zero-PE mu = {occ.mu:.4f}")
    width = fit.bin_centers[1] - fit.bin_centers[0]
    edges = np.append(fit.bin_centers - width / 2, fit.bin_centers[-1] + width / 2)
    ax.stairs(fit.counts, edges, label="signal window", color="C0")
    ax.hist(q_ctrl, bins=edges, histtype="step", color="0.5", label="control window")
    ax.plot(fit.bin_centers, fit.model, color="C3", lw=1.2, label="fit")
    ax.axvline(occ.threshold_pC, color="k", ls=":", lw=0.8, label="zero-PE threshold")
    ax.set_yscale("log")
    ax.set_ylim(0.5, None)
    ax.set_xlabel("charge (pC)")
    ax.set_ylabel("pulses / bin")
    ax.set_title(f"{PMTS[ch]}: Q1 = {fit.q1_pC:.3g} pC, gain {fit.gain:.3g}, mu = {fit.mu:.3f}",
                 fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(out_dir / f"spe_fit_{stamp}.png", dpi=150)
plt.show()

## Close

In [ ]:
ad3.stop_output(0)
ad3.close()
scope.close()

## Emergency: LED pulses off
Stops the AD3 output even if the cells above were not run (HV is untouched).

In [ ]:
try:
    ad3.stop_output(0)
    ad3.close()
except NameError:
    from ad3pulse.instrument import AnalogDiscovery
    with AnalogDiscovery() as _dev:
        _dev.stop_output(0)
print("AD3 output stopped")